## Silver Layer Cleaning Notebook

This notebook transforms Bronze data into a clean, standardised, and analysis-ready Silver layer for downstream modelling.

### Notebook Flow

1. Resolve project paths and validate Bronze source files.
2. Profile and clean users, cards, and transactions datasets.
3. Build cleaned reference tables for MCC and fraud labels.
4. Generate audit and column-profiling outputs.
5. Save all Silver outputs in parquet format.

In [10]:
# ======================================================
# Silver Cleaning Script
# Load latest Bronze datasets, clean them, and save Silver outputs.
# ======================================================

from datetime import datetime
from pathlib import Path
import json

import pandas as pd

current_dir = Path.cwd().resolve()
candidate_data_dirs = [
    current_dir / "Data",
    current_dir.parent / "Data",
]
data_dir = next((path for path in candidate_data_dirs if path.exists()), candidate_data_dirs[0])
if not data_dir.exists():
    checked = "\n - ".join(str(path) for path in candidate_data_dirs)
    raise FileNotFoundError(f"Data directory not found. Checked:\n - {checked}")

bronze_dir = data_dir / "bronze"
silver_dir = data_dir / "silver"
if not bronze_dir.exists():
    raise FileNotFoundError(f"Bronze directory not found: {bronze_dir}")
silver_dir.mkdir(parents=True, exist_ok=True)

print("Current working directory:", current_dir)
print("Data directory:", data_dir)
print("Bronze directory:", bronze_dir)
print("Silver directory:", silver_dir)

audit_log = []
column_profiles = []


def dq_score(df):
    total_cells = df.shape[0] * max(df.shape[1], 1)
    missing = df.isna().sum().sum()
    return round((1 - (missing / max(total_cells, 1))) * 100, 2)


def profile_columns(df, dataset):
    for col in df.columns:
        profile = {
            "dataset": dataset,
            "column": col,
            "dtype": str(df[col].dtype),
            "nulls": int(df[col].isna().sum()),
            "cardinality": int(df[col].nunique(dropna=True)),
        }
        if pd.api.types.is_numeric_dtype(df[col]):
            profile["min"] = float(df[col].min()) if not df[col].isna().all() else None
            profile["max"] = float(df[col].max()) if not df[col].isna().all() else None
        else:
            profile["min"] = None
            profile["max"] = None
        column_profiles.append(profile)


def audit(dataset, before_rows, after_rows, df, failed_dates=0, outliers=0):
    audit_log.append(
        {
            "timestamp": datetime.now(),
            "dataset": dataset,
            "rows_before": before_rows,
            "rows_after": after_rows,
            "duplicates_removed": before_rows - after_rows,
            "null_values": int(df.isna().sum().sum()),
            "failed_date_conversions": failed_dates,
            "outlier_rows": outliers,
            "dq_score_percent": dq_score(df),
        }
    )

Current working directory: C:\Users\9370892\OneDrive - Lloyds Banking Group\Apprenticeship - Data Science\Data Science Pro Practice\Assessment\Scripts
Data directory: C:\Users\9370892\OneDrive - Lloyds Banking Group\Apprenticeship - Data Science\Data Science Pro Practice\Assessment\Data
Bronze directory: C:\Users\9370892\OneDrive - Lloyds Banking Group\Apprenticeship - Data Science\Data Science Pro Practice\Assessment\Data\bronze
Silver directory: C:\Users\9370892\OneDrive - Lloyds Banking Group\Apprenticeship - Data Science\Data Science Pro Practice\Assessment\Data\silver


In [11]:
# ======================================================
# Data profiling helper
# ======================================================

def profile_dataset(df, name):
    print("=" * 50)
    print(f"{name.upper()} PROFILE")
    print("=" * 50)

    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")

    display(df.head())

    print("\nMissing values:")
    display(df.isna().sum().sort_values(ascending=False).head(10))

    print("\nData types:")
    display(df.dtypes)

In [12]:
bronze_patterns = {
    "transactions": "bronze_transactions_*.csv",
    "users": "bronze_users_*.csv",
    "cards": "bronze_cards_*.csv",
    "mcc": "bronze_mcc_codes_*.json",
    "labels": "bronze_fraud_labels_*.json",
}

def latest_file(directory, pattern):
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No files found for pattern '{pattern}' in {directory}")
    return matches[-1]

latest_transactions = latest_file(bronze_dir, bronze_patterns["transactions"])
latest_users = latest_file(bronze_dir, bronze_patterns["users"])
latest_cards = latest_file(bronze_dir, bronze_patterns["cards"])
latest_mcc = latest_file(bronze_dir, bronze_patterns["mcc"])
latest_labels = latest_file(bronze_dir, bronze_patterns["labels"])

print("Latest Bronze inputs:")
print(" - transactions:", latest_transactions.name)
print(" - users:", latest_users.name)
print(" - cards:", latest_cards.name)
print(" - mcc:", latest_mcc.name)
print(" - labels:", latest_labels.name)

Latest Bronze inputs:
 - transactions: bronze_transactions_20260824.csv
 - users: bronze_users_20260824.csv
 - cards: bronze_cards_20260824.csv
 - mcc: bronze_mcc_codes_20260824.json
 - labels: bronze_fraud_labels_20260824.json


In [13]:
# ======================================================
# USERS Profiling and Cleaning
# ======================================================

users = pd.read_csv(latest_users)

profile_dataset(users, 'users')

before = len(users)
users = users.drop_duplicates()
users.columns = users.columns.str.lower()
users = users.drop(columns=['address'], errors='ignore')

for c in ['per_capita_income', 'yearly_income', 'total_debt']:
    users[c] = users[c].replace('[$,]', '', regex=True).astype(float)

numeric_cols = [
    'per_capita_income',
    'yearly_income',
    'total_debt',
    'credit_score']
outlier_count = 0
for col in numeric_cols:
    q1 = users[col].quantile(0.25)
    q3 = users[col].quantile(0.75)
    iqr = q3 - q1
    users[f'{col}_outlier_flag'] = (
        (users[col] < (q1 - 1.5 * iqr)) | (users[col] > (q3 + 1.5 * iqr)))
    outlier_count += int(users[f'{col}_outlier_flag'].sum())

profile_columns(users, 'users')
audit('users', before, len(users), users, outliers=outlier_count)
users.to_parquet(silver_dir / 'silver_users.parquet', index=False)

USERS PROFILE
Rows: 2,000
Columns: 16


,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,snapshot_date,ingestion_timestamp
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5,20260824,2026-08-24T14:27:39
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5,20260824,2026-08-24T14:27:39
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5,20260824,2026-08-24T14:27:39
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4,20260824,2026-08-24T14:27:39
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1,20260824,2026-08-24T14:27:39



Missing values:


id                   0
current_age          0
retirement_age       0
birth_year           0
birth_month          0
gender               0
address              0
latitude             0
longitude            0
per_capita_income    0
dtype: int64


Data types:


id                       int64
current_age              int64
retirement_age           int64
birth_year               int64
birth_month              int64
gender                     str
address                    str
latitude               float64
longitude              float64
per_capita_income          str
yearly_income              str
total_debt                 str
credit_score             int64
num_credit_cards         int64
snapshot_date            int64
ingestion_timestamp        str
dtype: object

In [14]:
# ======================================================
# CARDS profiling and cleaning
# ======================================================

cards = pd.read_csv(latest_cards)

profile_dataset(cards, "cards")

before = len(cards)
cards = cards.drop_duplicates()
cards.columns = cards.columns.str.lower()
cards = cards.drop(columns=["card_number", "cvv"], errors="ignore")

cards["credit_limit"] = cards["credit_limit"].replace("[$,]", "", regex=True).astype(float)

cards["expires"] = pd.to_datetime(
    cards["expires"],
    format="%m/%Y",
    errors="coerce",
)
cards["acct_open_date"] = pd.to_datetime(
    cards["acct_open_date"],
    format="%m/%Y",
    errors="coerce",
)

failed_dates = int(cards["expires"].isna().sum() + cards["acct_open_date"].isna().sum())

cards["has_chip"] = cards["has_chip"].map({"YES": True, "NO": False})
cards["card_on_dark_web"] = cards["card_on_dark_web"].map({"Yes": True, "No": False})

profile_columns(cards, "cards")
audit("cards", before, len(cards), cards, failed_dates=failed_dates)

cards.to_parquet(silver_dir / "silver_cards.parquet", index=False)

CARDS PROFILE
Rows: 6,146
Columns: 15


,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,snapshot_date,ingestion_timestamp
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No,20260824,2026-08-24T14:27:39
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No,20260824,2026-08-24T14:27:39
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No,20260824,2026-08-24T14:27:39
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No,20260824,2026-08-24T14:27:39
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No,20260824,2026-08-24T14:27:39



Missing values:


id                  0
client_id           0
card_brand          0
card_type           0
card_number         0
expires             0
cvv                 0
has_chip            0
num_cards_issued    0
credit_limit        0
dtype: int64


Data types:


id                       int64
client_id                int64
card_brand                 str
card_type                  str
card_number              int64
expires                    str
cvv                      int64
has_chip                   str
num_cards_issued         int64
credit_limit               str
acct_open_date             str
year_pin_last_changed    int64
card_on_dark_web           str
snapshot_date            int64
ingestion_timestamp        str
dtype: object

In [15]:
# ======================================================
# Check date values and data type
# ======================================================
bronze_sample = pd.read_csv(
    latest_transactions,
    nrows=5
)

print(bronze_sample['date'])
print()
print(bronze_sample['date'].dtype)

0    2010-01-01 00:01:00
1    2010-01-01 00:02:00
2    2010-01-01 00:02:00
3    2010-01-01 00:05:00
4    2010-01-01 00:06:00
Name: date, dtype: str

str


In [16]:
# ======================================================
# TRANSACTIONS profiling and cleaning
# ======================================================

transactions = pd.read_csv(latest_transactions)

profile_dataset(transactions, "transactions")

before = len(transactions)
transactions = transactions.drop_duplicates()
transactions.columns = transactions.columns.str.lower()

transactions["date"] = pd.to_datetime(transactions["date"], errors="coerce")
failed_dates = int(transactions["date"].isna().sum())

transactions["amount"] = transactions["amount"].replace("[$,]", "", regex=True).astype(float)
transactions["is_refund"] = transactions["amount"] < 0

q1 = transactions["amount"].quantile(0.25)
q3 = transactions["amount"].quantile(0.75)
iqr = q3 - q1
transactions["amount_outlier_flag"] = (
    (transactions["amount"] < (q1 - 1.5 * iqr))
    | (transactions["amount"] > (q3 + 1.5 * iqr))
)
outliers = int(transactions["amount_outlier_flag"].sum())

profile_columns(transactions, "transactions")
audit(
    "transactions",
    before,
    len(transactions),
    transactions,
    failed_dates=failed_dates,
    outliers=outliers,
)

transactions.to_parquet(silver_dir / "silver_transactions.parquet", index=False)

TRANSACTIONS PROFILE
Rows: 13,305,915
Columns: 14


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,snapshot_date,ingestion_timestamp
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN,20260824,2026-08-24T14:27:39
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN,20260824,2026-08-24T14:27:39
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN,20260824,2026-08-24T14:27:39
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN,20260824,2026-08-24T14:27:39
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN,20260824,2026-08-24T14:27:39



Missing values:


errors            13094522
zip                1652706
merchant_state     1563700
client_id                0
id                       0
date                     0
use_chip                 0
amount                   0
card_id                  0
merchant_city            0
dtype: int64


Data types:


id                       int64
date                       str
client_id                int64
card_id                  int64
amount                     str
use_chip                   str
merchant_id              int64
merchant_city              str
merchant_state             str
zip                    float64
mcc                      int64
errors                     str
snapshot_date            int64
ingestion_timestamp        str
dtype: object

In [17]:
print(transactions.columns.tolist())
print(transactions.head())

['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'snapshot_date', 'ingestion_timestamp', 'is_refund', 'amount_outlier_flag']
        id                date  client_id  card_id  amount           use_chip  \
0  7475327 2010-01-01 00:01:00       1556     2972  -77.00  Swipe Transaction   
1  7475328 2010-01-01 00:02:00        561     4575   14.57  Swipe Transaction   
2  7475329 2010-01-01 00:02:00       1129      102   80.00  Swipe Transaction   
3  7475331 2010-01-01 00:05:00        430     2860  200.00  Swipe Transaction   
4  7475332 2010-01-01 00:06:00        848     3915   46.41  Swipe Transaction   

   merchant_id merchant_city merchant_state      zip   mcc errors  \
0        59935        Beulah             ND  58523.0  5499    NaN   
1        67570    Bettendorf             IA  52722.0  5311    NaN   
2        27092         Vista             CA  92084.0  4829    NaN   
3        27092   Crown Poi

In [18]:
transactions.isna().sum().sort_values(ascending=False)

errors                 13094522
zip                     1652706
merchant_state          1563700
id                            0
amount                        0
date                          0
client_id                     0
card_id                       0
merchant_city                 0
merchant_id                   0
use_chip                      0
mcc                           0
snapshot_date                 0
ingestion_timestamp           0
is_refund                     0
amount_outlier_flag           0
dtype: int64

In [19]:
# ======================================================
# MCC reference table
# ======================================================
with open(latest_mcc, encoding="utf-8") as file:
    mcc = json.load(file)
mcc_df = pd.DataFrame(mcc.items(), columns=["mcc", "merchant_category"])
mcc_df["mcc"] = mcc_df["mcc"].astype(int)
profile_columns(mcc_df, "mcc")
audit("mcc", len(mcc_df), len(mcc_df), mcc_df)
mcc_df.to_parquet(silver_dir / "silver_mcc_codes.parquet", index=False)

print(mcc_df.columns.tolist())

['mcc', 'merchant_category']


In [20]:
# ======================================================
# FRAUD LABELS reference table
# ======================================================
with open(latest_labels, encoding="utf-8") as file:
    labels = json.load(file)
if "target" not in labels:
    raise KeyError("Expected 'target' key in fraud labels JSON")

labels_df = pd.DataFrame.from_dict(labels["target"], orient="index").reset_index()
labels_df.columns = ["transaction_id", "fraud_label"]
labels_df["transaction_id"] = labels_df["transaction_id"].astype(int)
labels_df["is_fraud"] = labels_df["fraud_label"].map({"Yes": 1, "No": 0})
profile_columns(labels_df, "fraud_labels")
audit("fraud_labels", len(labels_df), len(labels_df), labels_df)
labels_df.to_parquet(silver_dir / "silver_fraud_labels.parquet", index=False)

In [21]:
# ======================================================
# AUDIT TABLES
# ======================================================
audit_df = pd.DataFrame(audit_log)
column_profiles_df = pd.DataFrame(column_profiles)

audit_df.to_parquet(silver_dir / "silver_audit_log.parquet", index=False)
column_profiles_df.to_parquet(
    silver_dir / "silver_column_profiles.parquet",
    index=False,
 )

print("AUDIT LOG")
print(audit_df)

print("COLUMN PROFILES")
print(column_profiles_df.head())

AUDIT LOG
                   timestamp       dataset  rows_before  rows_after  \
0 2026-08-24 14:45:40.896856         users         2000        2000   
1 2026-08-24 14:45:41.031419         cards         6146        6146   
2 2026-08-24 14:46:25.690954  transactions     13305915    13305915   
3 2026-08-24 14:46:31.241354           mcc          109         109   
4 2026-08-24 14:46:42.971572  fraud_labels      8914963     8914963   

   duplicates_removed  null_values  failed_date_conversions  outlier_rows  \
0                   0            0                        0           359   
1                   0            0                        0             0   
2                   0     16310928                        0       1052519   
3                   0            0                        0             0   
4                   0            0                        0             0   

   dq_score_percent  
0            100.00  
1            100.00  
2             92.34  
3           

In [22]:
print("Silver files generated:")
for file in sorted(silver_dir.glob("silver_*.parquet")):
    print(file.name)

print("MCC sample:")
display(mcc_df.head())

Silver files generated:
silver_audit_log.parquet
silver_cards.parquet
silver_column_profiles.parquet
silver_fraud_labels.parquet
silver_mcc_codes.parquet
silver_transactions.parquet
silver_users.parquet
MCC sample:


,mcc,merchant_category
0,5812,Eating Places and Restaurants
1,5541,Service Stations
2,7996,"Amusement Parks, Carnivals, Circuses"
3,5411,"Grocery Stores, Supermarkets"
4,4784,Tolls and Bridge Fees
